In [ ]:
"""
Script to download and prepare data from Kaggle
"""
import os
import sys
# Add project root to path

import pandas as pd
import numpy as np
import kagglehub
from sklearn.preprocessing import LabelEncoder
"""
Configuration for Multi-Modal VAE training
"""
import torch


class Config:
    """Training and model configuration"""
    
    # Model architecture
    INPUT_DIM_A = 1177  # RNA expression dimension @TODO
    INPUT_DIM_B = 1211  # DNA methylation dimension  @TODO
    LATENT_DIM = 20    # Latent space dimension 
    
    # Training parameters
    BATCH_SIZE = 32
    NUM_EPOCHS = 200
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-5
    
    # Loss parameters
    BETA_START = 1e-3  # KL divergence weight
    BETA_WARMUP_EPOCHS = 50  # Number of epochs for beta warmup
    GAMMA = 1.0  # Classification loss weight
    
    # Early stopping
    PATIENCE = 15
    
    # Optimizer
    LR_SCHEDULER_FACTOR = 0.5
    LR_SCHEDULER_PATIENCE = 5
    
    # Paths
    CHECKPOINT_DIR = 'checkpoints'
    BEST_MODEL_NAME = 'best_multivae.pt'
    
    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    # Data split
    TRAIN_TEST_SPLIT = 0.2
    RANDOM_SEED = 42


def download_datasets():
    """Download datasets from Kaggle"""
    print("Downloading RNA and mutations dataset...")
    rna_path = kagglehub.dataset_download('martininf1n1ty/plot-new-dataset')
    print(f"RNA dataset downloaded to: {rna_path}")
    
    print("\nDownloading DNA methylation dataset...")
    dna_path = kagglehub.dataset_download('martininf1n1ty/plot-new-dataset')
    print(f"DNA methylation dataset downloaded to: {dna_path}")
    
    return rna_path, dna_path

def prepare_rna_data(rna_path):
    """Prepare RNA expression data"""
    print("\nPreparing RNA expression data...")
    df_expressions = pd.read_parquet(f'{rna_path}/expression_new.parquet')
    
    # FIX: Ensure gene_name is a simple string and not a list/array
    # If gene_name is accidentally a list, this takes the first element
    if df_expressions['gene_name'].apply(lambda x: isinstance(x, (list, np.ndarray))).any():
        print("Warning: Nested arrays detected in gene_name. Extracting first elements...")
        df_expressions['gene_name'] = df_expressions['gene_name'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)

    # Sort and drop duplicates
    df_expressions_sorted = df_expressions.sort_values(by='gene_name')
    df_expressions_sorted = df_expressions_sorted.drop_duplicates(subset=['case_barcode', 'gene_name'])
    
    # Group by case_barcode and aggregate
    grouped_expressions_df = df_expressions_sorted.groupby('case_barcode').agg({
        'tpm_unstranded': list,
        'primary_site': 'first'
    }).reset_index()
    
    # Filter by expected dimension
    filtered_grouped_expressions_df = grouped_expressions_df[
        grouped_expressions_df['tpm_unstranded'].apply(len) == Config.INPUT_DIM_A
    ]
    
    print(f"RNA data shape: {filtered_grouped_expressions_df.shape}")
    return filtered_grouped_expressions_df




def prepare_dna_methylation_data(dna_path):
    """Prepare DNA methylation data"""
    print("\nPreparing DNA methylation data...")
    df = pd.read_parquet(f'/kaggle/input/plot-new-dataset/part-00000-d32c9147-df07-4faa-821d-7cc24149ac16-c000.snappy.parquet')
    
    # Sort by probe_id before grouping
    df_sorted = df.sort_values(by='probe_id_id')
    grouped_df = df_sorted.groupby('case_barcode')['beta_value'].apply(list).reset_index()

    filtered_grouped_methylation_df = grouped_df[
        grouped_df['beta_value'].apply(len) == Config.INPUT_DIM_B
    ]
    print(f"DNA methylation data shape: {filtered_grouped_methylation_df.shape}")
    return filtered_grouped_methylation_df


def merge_and_normalize_data(rna_df, dna_df, top_n_sites=24):
    """Merge all datasets and normalize"""
    print("\nMerging datasets...")
    
    # Merge RNA expression with DNA methylation using outer join to capture unmatched records
    merged_df = pd.merge(rna_df, dna_df, on='case_barcode', how='outer', indicator=True)
    
    # Identify and save unmatched records
    print("\nIdentifying unmatched records...")
    
    # RNA only (no matching DNA) - right side has NaN
    rna_only = merged_df[merged_df['_merge'] == 'left_only'].copy()
    if len(rna_only) > 0:
        print(f"Found {len(rna_only)} RNA samples without matching DNA methylation data")
        rna_only = rna_only[['case_barcode', 'tpm_unstranded', 'primary_site']]
        os.makedirs('data', exist_ok=True)
        rna_only.to_pickle('data/rna_only_unmatched.pkl')
        print(f"  Saved to: data/rna_only_unmatched.pkl")
    else:
        print("No RNA-only samples found")
    
    # DNA only (no matching RNA) - left side has NaN
    dna_only = merged_df[merged_df['_merge'] == 'right_only'].copy()
    if len(dna_only) > 0:
        print(f"Found {len(dna_only)} DNA methylation samples without matching RNA expression data")
        dna_only = dna_only[['case_barcode', 'beta_value']]
        dna_only.to_pickle('data/dna_only_unmatched.pkl')
        print(f"  Saved to: data/dna_only_unmatched.pkl")
    else:
        print("No DNA-only samples found")
    
    # Keep only successfully merged records
    merged_df = merged_df[merged_df['_merge'] == 'both'].copy()
    merged_df = merged_df.drop(columns=['_merge'])
    
    print(f"\nMerged data shape before filtering: {merged_df.shape}")
    
    # Filter to keep only top N most common primary sites
    print(f"\nFiltering to keep only top {top_n_sites} most common primary sites...")
    site_counts = merged_df['primary_site'].value_counts()
    print(f"Total number of unique primary sites: {len(site_counts)}")
    
    top_sites = site_counts.head(top_n_sites).index.tolist()
    print(f"\nTop {top_n_sites} primary sites:")
    for i, (site, count) in enumerate(site_counts.head(top_n_sites).items(), 1):
        print(f"  {i}. {site}: {count} samples")
    
    # Filter dataframe to keep only top sites
    merged_df = merged_df[merged_df['primary_site'].isin(top_sites)].reset_index(drop=True)
    print(f"\nMerged data shape after filtering: {merged_df.shape}")
    
    # Normalize tpm_unstranded data
    print("\nNormalizing RNA expression data...")
    merged_df["tpm_unstranded"] = merged_df["tpm_unstranded"].apply(
        lambda x: np.log1p(np.array(x))
    )
    
    # Encode primary site labels
    print("\nEncoding primary site labels...")
    label_encoder = LabelEncoder()
    merged_df['primary_site_encoded'] = label_encoder.fit_transform(merged_df['primary_site'])
    
    print(f"\nPrimary site encoding (all {len(label_encoder.classes_)} classes):")
    for cls, code in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
        count = (merged_df['primary_site'] == cls).sum()
        print(f"  {code}: {cls} ({count} samples)")
    
    return merged_df, label_encoder


def main():
    """Main data preparation pipeline"""
    # Download datasets
    rna_path, dna_path = download_datasets()
    
    # Prepare individual datasets
    rna_df = prepare_rna_data(rna_path)
    dna_df = prepare_dna_methylation_data(dna_path)
    
    # Merge and normalize
    merged_df, label_encoder = merge_and_normalize_data(rna_df, dna_df)
    
    # Save processed data
    print("\nSaving processed data...")
    os.makedirs('data', exist_ok=True)
    merged_df.to_pickle('data/processed_data.pkl')
    
    # Save label encoder
    import pickle
    with open('data/label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)
    
    print("\nData preparation complete!")
    print(f"Processed data saved to: data/processed_data.pkl")
    print(f"Label encoder saved to: data/label_encoder.pkl")
    print(f"\nAdditional files (if any unmatched records):")
    print(f"  - data/rna_only_unmatched.pkl (RNA samples without DNA)")
    print(f"  - data/dna_only_unmatched.pkl (DNA samples without RNA)")


if __name__ == "__main__":
    main()



RNA dataset downloaded to: /kaggle/input/plot-new-dataset

DNA methylation dataset downloaded to: /kaggle/input/plot-new-dataset

Preparing RNA expression data...
RNA data shape: (0, 3)

Preparing DNA methylation data...
DNA methylation data shape: (0, 2)

Merging datasets...

Identifying unmatched records...
No RNA-only samples found
No DNA-only samples found

Merged data shape before filtering: (0, 4)

Filtering to keep only top 24 most common primary sites...
Total number of unique primary sites: 0

Top 24 primary sites:

Merged data shape after filtering: (0, 4)

Normalizing RNA expression data...

Encoding primary site labels...

Primary site encoding (all 0 classes):

Saving processed data...

Data preparation complete!
Processed data saved to: data/processed_data.pkl
Label encoder saved to: data/label_encoder.pkl

Additional files (if any unmatched records):
  - data/rna_only_unmatched.pkl (RNA samples without DNA)
  - data/dna_only_unmatched.pkl (DNA samples without RNA)
